# How to create a source

This tutorial demonstrates how to create a source.

There are different type of sources available: luminaire source, surface source, ray file source.

## Prerequisites

### Perform imports

In [1]:
from pathlib import Path

from ansys.speos.core import Face, Project, Speos, launcher
from ansys.speos.core.generic.parameters import CieType
from ansys.speos.core.kernel.client import (
    default_docker_channel,
)
from ansys.speos.core.source import (
    SourceAmbientCieStandardGeneralSky,
    SourceAmbientCieStandardOvercastSky,
    SourceAmbientEnvironment,
    SourceAmbientNaturalLight,
    SourceAmbientUniform,
    SourceAmbientUsStandard,
    SourceDisplay,
    SourceLuminaire,
    SourceRayFile,
    SourceSurface,
    SourceThermic,
)


### Define constants
Constants help ensure consistency and avoid repetition throughout the example.

In [2]:
HOSTNAME = "localhost"
GRPC_PORT = 50098  # Be sure the Speos GRPC Server has been started on this port.
USE_DOCKER = True  # Set to False if you're running this example locally as a Notebook.
IES = "IES_C_DETECTOR.ies"

### Define helper functions

In [3]:
def create_helper_geometries(project: Project):
    """Create bodies and faces."""

    def create_face(body):
        f = body.create_face(name="TheFaceF")
        f.vertices = [0, 0, 0, 1, 0, 0, 0, 1, 0]
        f.facets = [0, 1, 2]
        f.normals = [0, 0, 1, 0, 0, 1, 0, 0, 1]
        f.commit()

    root_part = project.create_root_part().commit()
    body_b1 = root_part.create_body(name="TheBodyB").commit()
    body_b2 = root_part.create_body(name="TheBodyC").commit()
    body_b3 = root_part.create_body(name="TheBodyD").commit()
    body_b4 = root_part.create_body(name="TheBodyE").commit()
    for b in [body_b1, body_b2, body_b3, body_b4]:
        create_face(b)

## Model Setup

### Load assets
The assets used to run this example are available in the
[PySpeos repository](https://github.com/ansys/pyspeos/) on GitHub.

> **Note:** Make sure you
> have downloaded simulation assets and set ``assets_data_path``
> to point to the assets folder.

In [4]:
if USE_DOCKER:  # Running on the remote server.
    assets_data_path = Path("/app") / "assets"
else:
    assets_data_path = Path("/path/to/your/download/assets/directory")

### Connect to the RPC Server
This Python client connects to a server where the Speos engine
is running as a service. In this example, the server and
client are the same machine. The launch_local_speos_rpc_method can
be used to start a local instance of the service.

In [5]:
if USE_DOCKER:
    speos = Speos(channel=default_docker_channel())
else:
    speos = launcher.launch_local_speos_rpc_server(port=GRPC_PORT)

/home/runner/work/pyspeos/pyspeos/.venv/lib/python3.14/site-packages/ansys/tools/common/cyberchannel.py:201: UserWarning: Starting gRPC client without TLS on localhost:50098. This is INSECURE. Consider using a secure connection.
  warn(f"Starting gRPC client without TLS on {target}. This is INSECURE. Consider using a secure connection.")


### Create a new project

The only way to create a source using the core layer, is to create it from a project.
The ``Project`` class is instantiated by passing a ``Speos`` instance

In [6]:
p = Project(speos=speos)
print(p)

{
    "name": "",
    "description": "",
    "metadata": {},
    "part_guid": "",
    "sub_scene_anchor_axis_system": [],
    "sources": [],
    "sensors": [],
    "simulations": [],
    "materials": [],
    "scenes": [],
    "textures_3d": []
}


### Source Creation

**Create locally:**
The mention "local: " is added when printing the source data and information is not yet
pushed to the RPC server

In [7]:
intensity_file_path = assets_data_path / IES
source1 = p.create_source(name="Luminaire.1", feature_type=SourceLuminaire)  # type luminaire
source1.intensity_file_uri = intensity_file_path
print(source1)

local: {
    "name": "Luminaire.1",
    "display_name": "",
    "description": "",
    "metadata": {},
    "source_guid": "",
    "source": {
        "name": "Luminaire.1",
        "luminaire": {
            "flux_from_intensity_file": {},
            "intensity_file_uri": "/app/assets/IES_C_DETECTOR.ies",
            "spectrum_guid": "",
            "axis_system": [
                0.0,
                0.0,
                0.0,
                1.0,
                0.0,
                0.0,
                0.0,
                1.0,
                0.0,
                0.0,
                0.0,
                1.0
            ]
        },
        "description": "",
        "metadata": {}
    }
}


**Push it to the server.**

After it is committed to the server, the mention "local: " is no more present when printing the
source.

In [8]:
source1.commit()
print(source1)

{
    "name": "Luminaire.1",
    "metadata": {
        "UniqueId": "f7367ad3-9d2f-48e3-8bb0-9324968b1828"
    },
    "source_guid": "c7dc8887-f096-4c67-bc0b-b9a00b26e35f",
    "display_name": "",
    "description": "",
    "source": {
        "name": "Luminaire.1",
        "luminaire": {
            "flux_from_intensity_file": {},
            "intensity_file_uri": "/app/assets/IES_C_DETECTOR.ies",
            "spectrum_guid": "9ab0543b-4a29-46e6-918b-1d4cbc8009f7",
            "spectrum": {
                "name": "Luminaire.1.Spectrum",
                "predefined": {
                    "incandescent": {}
                },
                "description": "",
                "metadata": {}
            },
            "axis_system": [
                0.0,
                0.0,
                0.0,
                1.0,
                0.0,
                0.0,
                0.0,
                1.0,
                0.0,
                0.0,
                0.0,
                1.0
     

**Changing additional Source Properties**

Setting several more characteristics.

In [9]:
intensity_file_path = assets_data_path / IES
source2 = p.create_source(name="Luminaire.2", feature_type=SourceLuminaire)
source2.intensity_file_uri = intensity_file_path
source2.flux.set_radiant()
source2.flux.value = 1.0  # select flux radiant
# choose the source location [Origin, Xvector, Yvector, Zvector]
source2.axis_system = [20, 50, 10, 1, 0, 0, 0, 1, 0, 0, 0, 1]
source2.spectrum.set_blackbody()  # choose blackbody with default value for the spectrum
source2.commit()  # Push to the server
print(source2)

{
    "name": "Luminaire.2",
    "metadata": {
        "UniqueId": "57c553b5-7397-434f-8187-4f40c8123771"
    },
    "source_guid": "b84d328a-d58a-4b39-bdf9-26348f18696d",
    "display_name": "",
    "description": "",
    "source": {
        "name": "Luminaire.2",
        "luminaire": {
            "radiant_flux": {
                "radiant_value": 1.0
            },
            "intensity_file_uri": "/app/assets/IES_C_DETECTOR.ies",
            "spectrum_guid": "60576953-13e4-40c7-93dd-5a8d1c900872",
            "spectrum": {
                "name": "Luminaire.2.Spectrum",
                "blackbody": {
                    "temperature": 2856.0
                },
                "description": "",
                "metadata": {}
            },
            "axis_system": [
                20.0,
                50.0,
                10.0,
                1.0,
                0.0,
                0.0,
                0.0,
                1.0,
                0.0,
                0.0,
   

**Source Instance**

As mention "local: " is added if it is not yet committed to the server.

In [10]:
print(source1)

{
    "name": "Luminaire.1",
    "metadata": {
        "UniqueId": "f7367ad3-9d2f-48e3-8bb0-9324968b1828"
    },
    "source_guid": "c7dc8887-f096-4c67-bc0b-b9a00b26e35f",
    "display_name": "",
    "description": "",
    "source": {
        "name": "Luminaire.1",
        "luminaire": {
            "flux_from_intensity_file": {},
            "intensity_file_uri": "/app/assets/IES_C_DETECTOR.ies",
            "spectrum_guid": "9ab0543b-4a29-46e6-918b-1d4cbc8009f7",
            "spectrum": {
                "name": "Luminaire.1.Spectrum",
                "predefined": {
                    "incandescent": {}
                },
                "description": "",
                "metadata": {}
            },
            "axis_system": [
                0.0,
                0.0,
                0.0,
                1.0,
                0.0,
                0.0,
                0.0,
                1.0,
                0.0,
                0.0,
                0.0,
                1.0
     

**Project:**

Committed feature will appear inside the project information.

In [11]:
print(p)

{
    "sources": [
        {
            "name": "Luminaire.1",
            "metadata": {
                "UniqueId": "f7367ad3-9d2f-48e3-8bb0-9324968b1828"
            },
            "source_guid": "c7dc8887-f096-4c67-bc0b-b9a00b26e35f",
            "display_name": "",
            "description": "",
            "source": {
                "name": "Luminaire.1",
                "luminaire": {
                    "flux_from_intensity_file": {},
                    "intensity_file_uri": "/app/assets/IES_C_DETECTOR.ies",
                    "spectrum_guid": "9ab0543b-4a29-46e6-918b-1d4cbc8009f7",
                    "spectrum": {
                        "name": "Luminaire.1.Spectrum",
                        "predefined": {
                            "incandescent": {}
                        },
                        "description": "",
                        "metadata": {}
                    },
                    "axis_system": [
                        0.0,
                        

**Update:**

> **Note:** If you are manipulating a source already committed, don't forget to commit your
> changes.
> If you don't, you will still only watch what is committed on the server.

In [12]:
source1.flux.set_radiant()
source1.flux.value = 1.2  # modify radiant flux value
source1.axis_system = [17, 10, 10, 1, 0, 0, 0, 1, 0, 0, 0, 1]  # modify axis system
source1.spectrum.set_halogen()  # modify spectrum by choosing halogen
source1.commit()  # Push changes to the server
print(source1)

{
    "name": "Luminaire.1",
    "metadata": {
        "UniqueId": "f7367ad3-9d2f-48e3-8bb0-9324968b1828"
    },
    "source_guid": "c7dc8887-f096-4c67-bc0b-b9a00b26e35f",
    "display_name": "",
    "description": "",
    "source": {
        "name": "Luminaire.1",
        "luminaire": {
            "radiant_flux": {
                "radiant_value": 1.2
            },
            "intensity_file_uri": "/app/assets/IES_C_DETECTOR.ies",
            "spectrum_guid": "9ab0543b-4a29-46e6-918b-1d4cbc8009f7",
            "spectrum": {
                "name": "Luminaire.1.Spectrum",
                "predefined": {
                    "halogen": {}
                },
                "description": "",
                "metadata": {}
            },
            "axis_system": [
                17.0,
                10.0,
                10.0,
                1.0,
                0.0,
                0.0,
                0.0,
                1.0,
                0.0,
                0.0,
          

**Reset**

Possibility to reset local values from the one available in the server.

In [13]:
source1.flux.set_luminous().value = 683.0  # modify to luminous flux BUT no commit
source1.reset()
# reset -> this will apply the server value to the local value the local value will be back to
# halogen
source1.delete()  # delete (to display the local value with the below print)
print(source1)

local: {
    "name": "Luminaire.1",
    "display_name": "",
    "description": "",
    "metadata": {},
    "source_guid": "",
    "source": {
        "name": "Luminaire.1",
        "luminaire": {
            "radiant_flux": {
                "radiant_value": 1.2
            },
            "intensity_file_uri": "/app/assets/IES_C_DETECTOR.ies",
            "spectrum_guid": "9ab0543b-4a29-46e6-918b-1d4cbc8009f7",
            "spectrum": {
                "name": "Luminaire.1.Spectrum",
                "predefined": {
                    "halogen": {}
                },
                "description": "",
                "metadata": {}
            },
            "axis_system": [
                17.0,
                10.0,
                10.0,
                1.0,
                0.0,
                0.0,
                0.0,
                1.0,
                0.0,
                0.0,
                0.0,
                1.0
            ]
        },
        "description": "",
        "m

**Delete**

Once the data is deleted from the server, you can still work with local data and maybe commit
later.

In [14]:
source2.delete()
print(source2)
source1.delete()
print(p)

local: {
    "name": "Luminaire.2",
    "display_name": "",
    "description": "",
    "metadata": {},
    "source_guid": "",
    "source": {
        "name": "Luminaire.2",
        "luminaire": {
            "radiant_flux": {
                "radiant_value": 1.0
            },
            "intensity_file_uri": "/app/assets/IES_C_DETECTOR.ies",
            "spectrum_guid": "60576953-13e4-40c7-93dd-5a8d1c900872",
            "spectrum": {
                "name": "Luminaire.2.Spectrum",
                "blackbody": {
                    "temperature": 2856.0
                },
                "description": "",
                "metadata": {}
            },
            "axis_system": [
                20.0,
                50.0,
                10.0,
                1.0,
                0.0,
                0.0,
                0.0,
                1.0,
                0.0,
                0.0,
                0.0,
                1.0
            ]
        },
        "description": "",
   

## Other Sources Examples

### Ray-file source

In [15]:
ray_file_path = assets_data_path / "Rays.ray"

source3 = p.create_source(name="Ray-file.1", feature_type=SourceRayFile)  # type ray file
source3.ray_file_uri = ray_file_path
source3.commit()
print(source3)

{
    "name": "Ray-file.1",
    "metadata": {
        "UniqueId": "ef4a3c2c-a1bb-47a8-a156-b267c50c3a19"
    },
    "source_guid": "f6e99a56-a097-49a4-b57f-c6cd70b2733e",
    "display_name": "",
    "description": "",
    "source": {
        "name": "Ray-file.1",
        "rayfile": {
            "ray_file_uri": "/app/assets/Rays.ray",
            "flux_from_ray_file": {},
            "spectrum_from_ray_file": {},
            "axis_system": [
                0.0,
                0.0,
                0.0,
                1.0,
                0.0,
                0.0,
                0.0,
                1.0,
                0.0,
                0.0,
                0.0,
                1.0
            ]
        },
        "description": "",
        "metadata": {}
    }
}


In [16]:
source3.flux.set_luminous()
source3.flux.value = 55
source3.commit()
print(source3)

{
    "name": "Ray-file.1",
    "metadata": {
        "UniqueId": "ef4a3c2c-a1bb-47a8-a156-b267c50c3a19"
    },
    "source_guid": "f6e99a56-a097-49a4-b57f-c6cd70b2733e",
    "display_name": "",
    "description": "",
    "source": {
        "name": "Ray-file.1",
        "rayfile": {
            "ray_file_uri": "/app/assets/Rays.ray",
            "luminous_flux": {
                "luminous_value": 55.0
            },
            "spectrum_from_ray_file": {},
            "axis_system": [
                0.0,
                0.0,
                0.0,
                1.0,
                0.0,
                0.0,
                0.0,
                1.0,
                0.0,
                0.0,
                0.0,
                1.0
            ]
        },
        "description": "",
        "metadata": {}
    }
}


In [17]:
source3.delete()

### Surface source

In [18]:
create_helper_geometries(p)
source4 = p.create_source(name="Surface.1", feature_type=SourceSurface)
face1 = p.find(name="TheBodyB/TheFaceF", feature_type=Face)[0]
face2 = p.find(name="TheBodyC/TheFaceF", feature_type=Face)[0]
source4.set_exitance_constant().geometries = [
    (face1, False),
    (face2, True),
]
source4.commit()
print(source4)

{
    "name": "Surface.1",
    "metadata": {
        "UniqueId": "4d6d0efb-82e5-4361-bb9b-c9390a571846"
    },
    "source_guid": "eff59761-80c7-4935-9e83-ad364eccffce",
    "display_name": "",
    "description": "",
    "source": {
        "name": "Surface.1",
        "surface": {
            "luminous_flux": {
                "luminous_value": 683.0
            },
            "intensity_guid": "1c8721ef-fda5-472c-9d42-98c505dac871",
            "exitance_constant": {
                "geo_paths": [
                    {
                        "geo_path": "TheBodyB/TheFaceF",
                        "reverse_normal": false
                    },
                    {
                        "geo_path": "TheBodyC/TheFaceF",
                        "reverse_normal": true
                    }
                ]
            },
            "spectrum_guid": "b94ce36a-366c-4046-a067-9dccf36acb12",
            "intensity": {
                "name": "Surface.1.Intensity",
                "cos"

In [19]:
source4.flux.set_luminous_intensity()
source4.flux.value = 6
source4.intensity.set_gaussian().axis_system = [10, 50, 20, 1, 0, 0, 0, 1, 0, 0, 0, 1]
source4.commit()
print(source4)

{
    "name": "Surface.1",
    "metadata": {
        "UniqueId": "4d6d0efb-82e5-4361-bb9b-c9390a571846"
    },
    "source_guid": "eff59761-80c7-4935-9e83-ad364eccffce",
    "display_name": "",
    "description": "",
    "source": {
        "name": "Surface.1",
        "surface": {
            "luminous_intensity_flux": {
                "luminous_intensity_value": 6.0
            },
            "intensity_guid": "1c8721ef-fda5-472c-9d42-98c505dac871",
            "exitance_constant": {
                "geo_paths": [
                    {
                        "geo_path": "TheBodyB/TheFaceF",
                        "reverse_normal": false
                    },
                    {
                        "geo_path": "TheBodyC/TheFaceF",
                        "reverse_normal": true
                    }
                ]
            },
            "spectrum_guid": "b94ce36a-366c-4046-a067-9dccf36acb12",
            "intensity": {
                "name": "Surface.1.Intensity",
   

In [20]:
source4.delete()
print(source4)

local: {
    "name": "Surface.1",
    "display_name": "",
    "description": "",
    "metadata": {},
    "source_guid": "",
    "source": {
        "name": "Surface.1",
        "surface": {
            "luminous_intensity_flux": {
                "luminous_intensity_value": 6.0
            },
            "intensity_guid": "1c8721ef-fda5-472c-9d42-98c505dac871",
            "exitance_constant": {
                "geo_paths": [
                    {
                        "geo_path": "TheBodyB/TheFaceF",
                        "reverse_normal": false
                    },
                    {
                        "geo_path": "TheBodyC/TheFaceF",
                        "reverse_normal": true
                    }
                ]
            },
            "spectrum_guid": "b94ce36a-366c-4046-a067-9dccf36acb12",
            "intensity": {
                "name": "Surface.1.Intensity",
                "gaussian": {
                    "FWHM_angle_x": 30.0,
                    "FWH

### Ambient natural light source

In [21]:
source5 = p.create_source(name="NaturalLight.1", feature_type=SourceAmbientNaturalLight)
source5.turbidity = 4
source5.with_sky = True
print(source5.zenith_direction)  # default zenith direction
print(source5.north_direction)  # default north direction
source5.reverse_north_direction = True
print(source5)

source5.commit()
print(source5)

[0.0, 0.0, 1.0]
[0.0, 1.0, 0.0]
local: {
    "name": "NaturalLight.1",
    "display_name": "",
    "description": "",
    "metadata": {},
    "source_guid": "",
    "source": {
        "name": "NaturalLight.1",
        "ambient": {
            "natural_light": {
                "turbidity": 4.0,
                "with_sky": true,
                "north_direction": [
                    0.0,
                    1.0,
                    0.0
                ],
                "reverse_north": true,
                "sun_axis_system": {
                    "automatic_sun": {
                        "time_zone_uri": "CET",
                        "year": 2026,
                        "month": 11,
                        "day": 12,
                        "hour": 16,
                        "minute": 24,
                        "longitude": 0.0,
                        "latitude": 0.0
                    }
                }
            },
            "zenith_direction": [
                0.0,


{
    "name": "NaturalLight.1",
    "metadata": {
        "UniqueId": "d959f74b-49c1-463b-a47b-b0105c1911ea"
    },
    "source_guid": "f76d6d69-be2b-454b-9bba-fb4b428b23f3",
    "display_name": "",
    "description": "",
    "source": {
        "name": "NaturalLight.1",
        "ambient": {
            "natural_light": {
                "turbidity": 4.0,
                "with_sky": true,
                "north_direction": [
                    0.0,
                    1.0,
                    0.0
                ],
                "reverse_north": true,
                "sun_axis_system": {
                    "automatic_sun": {
                        "time_zone_uri": "CET",
                        "year": 2026,
                        "month": 11,
                        "day": 12,
                        "hour": 16,
                        "minute": 24,
                        "longitude": 0.0,
                        "latitude": 0.0
                    }
                }
         

In [22]:
source5.set_sun_automatic().year = 2026
source5.set_sun_automatic().month = 12
source5.set_sun_automatic().day = 31
source5.set_sun_automatic().hour = 12
source5.set_sun_automatic().minute = 23
source5.set_sun_automatic().longitude = 10
source5.set_sun_automatic().latitude = 45
source5.set_sun_automatic().time_zone = "CST"
source5.commit()
print(source5)

{
    "name": "NaturalLight.1",
    "metadata": {
        "UniqueId": "d959f74b-49c1-463b-a47b-b0105c1911ea"
    },
    "source_guid": "f76d6d69-be2b-454b-9bba-fb4b428b23f3",
    "display_name": "",
    "description": "",
    "source": {
        "name": "NaturalLight.1",
        "ambient": {
            "natural_light": {
                "turbidity": 4.0,
                "with_sky": true,
                "north_direction": [
                    0.0,
                    1.0,
                    0.0
                ],
                "reverse_north": true,
                "sun_axis_system": {
                    "automatic_sun": {
                        "time_zone_uri": "CST",
                        "year": 2026,
                        "month": 12,
                        "day": 31,
                        "hour": 12,
                        "minute": 23,
                        "longitude": 10.0,
                        "latitude": 45.0
                    }
                }
       

In [23]:
source5.delete()

### Ambient uniform light source

In [24]:
ambient_uniform_source = p.create_source(name="Uniform.1", feature_type=SourceAmbientUniform)
print(ambient_uniform_source.luminance)  # default luminance value
print(ambient_uniform_source.zenith_direction)  # default zenith direction
ambient_uniform_source.mirrored_extent = True
ambient_uniform_source.luminance = 500.0
ambient_uniform_source.zenith_direction = [0, 1, 0]
ambient_uniform_source.reverse_zenith_direction = True
ambient_uniform_source.spectrum.set_blackbody().temperature = 5500
ambient_uniform_source.commit()
print(ambient_uniform_source)

1000.0
[0.0, 0.0, 1.0]
{
    "name": "Uniform.1",
    "metadata": {
        "UniqueId": "63809dfd-d91c-4b12-881b-a67c0b14812e"
    },
    "source_guid": "4d73a2c7-e44b-4bc1-94d3-55ea37b45eba",
    "display_name": "",
    "description": "",
    "source": {
        "name": "Uniform.1",
        "ambient": {
            "uniform_ambient": {
                "luminance": 500.0,
                "mirrored_extent": true,
                "spectrum_guid": "bc25a195-7b8b-4b97-a01a-eae55a032c72",
                "spectrum": {
                    "name": "Uniform.1.Spectrum",
                    "blackbody": {
                        "temperature": 5500.0
                    },
                    "description": "",
                    "metadata": {}
                }
            },
            "zenith_direction": [
                0.0,
                1.0,
                0.0
            ],
            "reverse_zenith": true
        },
        "description": "",
        "metadata": {}
    }
}


In [25]:
ambient_uniform_source.set_sun_manual().direction = [1, 0, 0]
ambient_uniform_source.set_sun_manual().reverse_sun = True
ambient_uniform_source.commit()
print(ambient_uniform_source)

{
    "name": "Uniform.1",
    "metadata": {
        "UniqueId": "63809dfd-d91c-4b12-881b-a67c0b14812e"
    },
    "source_guid": "4d73a2c7-e44b-4bc1-94d3-55ea37b45eba",
    "display_name": "",
    "description": "",
    "source": {
        "name": "Uniform.1",
        "ambient": {
            "uniform_ambient": {
                "luminance": 500.0,
                "mirrored_extent": true,
                "spectrum_guid": "bc25a195-7b8b-4b97-a01a-eae55a032c72",
                "spectrum": {
                    "name": "Uniform.1.Spectrum",
                    "blackbody": {
                        "temperature": 5500.0
                    },
                    "description": "",
                    "metadata": {}
                },
                "manual_sun": {
                    "sun_direction": [
                        1.0,
                        0.0,
                        0.0
                    ],
                    "reverse_sun": true
                }
            },
    

In [26]:
ambient_uniform_source.delete()

### Ambient CIE Standard General Sky light source

In [27]:
ambient_cie_standard_general_sky_source = p.create_source(
    name="CieStandardGeneralSky.1", feature_type=SourceAmbientCieStandardGeneralSky
)
print(ambient_cie_standard_general_sky_source.luminance)  # default luminance value
print(ambient_cie_standard_general_sky_source.zenith_direction)  # default zenith direction
print(ambient_cie_standard_general_sky_source.north_direction)  # default north direction
print(ambient_cie_standard_general_sky_source.cie_type)  # default cie setting
ambient_cie_standard_general_sky_source.luminance = 500.0
ambient_cie_standard_general_sky_source.north_direction = [1, 0, 0]
ambient_cie_standard_general_sky_source.reverse_north_direction = True
ambient_cie_standard_general_sky_source.cie_type = CieType.standard_overcast
ambient_cie_standard_general_sky_source.commit()
print(ambient_cie_standard_general_sky_source)

1000.0
[0.0, 0.0, 1.0]
[0.0, 1.0, 0.0]
CieType.cloudy_slight_brightening


{
    "name": "CieStandardGeneralSky.1",
    "metadata": {
        "UniqueId": "79dc1ee5-8ea0-46c4-81a9-ca5afb6dd807"
    },
    "source_guid": "298971f8-a3b8-46e5-a891-1e7ac7fe6b32",
    "display_name": "",
    "description": "",
    "source": {
        "name": "CieStandardGeneralSky.1",
        "ambient": {
            "cie_general": {
                "luminance": 500.0,
                "cie_type": "standard_overcast",
                "north_direction": [
                    1.0,
                    0.0,
                    0.0
                ],
                "reverse_north": true,
                "sun_axis_system": {
                    "automatic_sun": {
                        "time_zone_uri": "CET",
                        "year": 2026,
                        "month": 11,
                        "day": 12,
                        "hour": 16,
                        "minute": 24,
                        "longitude": 0.0,
                        "latitude": 0.0
                

In [28]:
ambient_cie_standard_general_sky_source.set_sun_manual().direction = [0, 0.707, 0.707]
ambient_cie_standard_general_sky_source.set_sun_manual().reverse_sun = True
ambient_cie_standard_general_sky_source.commit()
print(ambient_cie_standard_general_sky_source)

{
    "name": "CieStandardGeneralSky.1",
    "metadata": {
        "UniqueId": "79dc1ee5-8ea0-46c4-81a9-ca5afb6dd807"
    },
    "source_guid": "298971f8-a3b8-46e5-a891-1e7ac7fe6b32",
    "display_name": "",
    "description": "",
    "source": {
        "name": "CieStandardGeneralSky.1",
        "ambient": {
            "cie_general": {
                "luminance": 500.0,
                "cie_type": "standard_overcast",
                "north_direction": [
                    1.0,
                    0.0,
                    0.0
                ],
                "reverse_north": true,
                "sun_axis_system": {
                    "manual_sun": {
                        "sun_direction": [
                            0.0,
                            0.707,
                            0.707
                        ],
                        "reverse_sun": true
                    }
                }
            },
            "zenith_direction": [
                0.0,
       

In [29]:
ambient_cie_standard_general_sky_source.delete()

### Ambient U.S. Standard light source

In [30]:
ambient_us_standard_source = p.create_source(
    name="UsStandard.1", feature_type=SourceAmbientUsStandard
)
print(ambient_us_standard_source.zenith_direction)  # default zenith direction
print(ambient_us_standard_source.north_direction)  # default north direction
ambient_us_standard_source.zenith_direction = [0, 1, 0]
ambient_us_standard_source.north_direction = [1, 0, 0]
ambient_us_standard_source.reverse_north_direction = True
ambient_us_standard_source.set_sun_manual().direction = [0, 0.707, 0.707]
ambient_us_standard_source.commit()
print(ambient_us_standard_source)

[0.0, 0.0, 1.0]
[0.0, 1.0, 0.0]
{
    "name": "UsStandard.1",
    "metadata": {
        "UniqueId": "043509a1-07d1-42f0-9390-fb5bf3cc88b6"
    },
    "source_guid": "1432b13c-b011-4539-9e88-0abdb118af07",
    "display_name": "",
    "description": "",
    "source": {
        "name": "UsStandard.1",
        "ambient": {
            "us_standard": {
                "north_direction": [
                    1.0,
                    0.0,
                    0.0
                ],
                "reverse_north": true,
                "sun_axis_system": {
                    "manual_sun": {
                        "sun_direction": [
                            0.0,
                            0.707,
                            0.707
                        ],
                        "reverse_sun": false
                    }
                }
            },
            "zenith_direction": [
                0.0,
                1.0,
                0.0
            ],
            "reverse_zeni

In [31]:
ambient_us_standard_source.delete()
# ### Ambient CIE overcast light source

In [32]:
ambient_cie_overcast_source = p.create_source(
    name="CieOvercast.1", feature_type=SourceAmbientCieStandardOvercastSky
)
print(ambient_cie_overcast_source.luminance)  # default luminance value
print(ambient_cie_overcast_source.zenith_direction)  # default zenith direction
print(ambient_cie_overcast_source.spectrum.set_blackbody().temperature)  # default spectrum
ambient_cie_overcast_source.luminance = 750.0
ambient_cie_overcast_source.zenith_direction = [0, 1, 0]
ambient_cie_overcast_source.reverse_zenith_direction = True
ambient_cie_overcast_source.spectrum.set_blackbody().temperature = 4500
ambient_cie_overcast_source.commit()
print(ambient_cie_overcast_source)

1000.0
[0.0, 0.0, 1.0]
2856.0
{
    "name": "CieOvercast.1",
    "metadata": {
        "UniqueId": "bb68bd84-1a54-475b-950a-7e97e79a4057"
    },
    "source_guid": "ce29bc51-7fc9-44e4-8261-96ac13fe5024",
    "display_name": "",
    "description": "",
    "source": {
        "name": "CieOvercast.1",
        "ambient": {
            "cie_overcast": {
                "luminance": 750.0,
                "spectrum_guid": "68f6cbe5-1760-4e27-9cd0-7bb8ed2cce73",
                "spectrum": {
                    "name": "CieOvercast.1.Spectrum",
                    "blackbody": {
                        "temperature": 4500.0
                    },
                    "description": "",
                    "metadata": {}
                }
            },
            "zenith_direction": [
                0.0,
                1.0,
                0.0
            ],
            "reverse_zenith": true
        },
        "description": "",
        "metadata": {}
    }
}


In [33]:
ambient_cie_overcast_source.delete()

### Ambient environment light source

In [34]:
source6 = p.create_source(name="Environment.1", feature_type=SourceAmbientEnvironment)
image_file_uri = assets_data_path / "stars.exr"
source6.image_file_uri = image_file_uri
print(source6.zenith_direction)  # default zenith direction
print(source6.north_direction)  # default north direction
source6.reverse_north_direction = True
print(source6.luminance)  # default luminance value
source6.set_predefined_color_space().set_color_space_adobergb()
source6.set_predefined_color_space().set_color_space_srgb()
print(source6.color_space)
source6.set_userdefined_color_space().set_white_point_type_d50()
print(source6.set_userdefined_color_space().white_point_type)
source6.set_userdefined_color_space().red_spectrum = (
    assets_data_path / "LG_50M_Colorimetric_short.sv5" / "Red Spectrum.spectrum"
)
source6.set_userdefined_color_space().blue_spectrum = (
    assets_data_path / "LG_50M_Colorimetric_short.sv5" / "Blue Spectrum.spectrum"
)
source6.set_userdefined_color_space().green_spectrum = (
    assets_data_path / "LG_50M_Colorimetric_short.sv5" / "Blue Spectrum.spectrum"
)
source6.commit()
print(source6)

[0.0, 0.0, 1.0]
[0.0, 1.0, 0.0]
1000.0
1


{
    "name": "Environment.1",
    "metadata": {
        "UniqueId": "b5e637ba-3a8c-4733-b752-49a5b25349ee"
    },
    "source_guid": "0b41a621-3956-4c88-8751-f27282453d59",
    "display_name": "",
    "description": "",
    "source": {
        "name": "Environment.1",
        "ambient": {
            "environment_map": {
                "luminance": 1000.0,
                "image_uri": "/app/assets/stars.exr",
                "user_defined_rgb_space": {
                    "pre_defined_white_point": {
                        "white_point_type": "D50"
                    },
                    "red_spectrum_guid": "de4b4474-140e-4cb7-8cfd-ffae033e6199",
                    "green_spectrum_guid": "3e59d0c9-0f78-4a94-9e9f-f63c1fd9d763",
                    "blue_spectrum_guid": "5c82d5f8-322d-4312-bf08-8fb782ed1e9b",
                    "red_spectrum": {
                        "name": "Red Spectrum.spectrum",
                        "library": {
                            "file_uri": "

In [35]:
ground_plane = (
    p.create_ground_plane()
)  #  Only usable when there is at least one Ambient Environment Source
ground_plane.ground_zenith = [0, 0, 1]
ground_plane.ground_origin = [10, 0, 0]
ground_plane.ground_height = 1000
ground_plane.commit()
print(p)

ground_plane2 = p.create_ground_plane()
ground_plane2.ground_height = 2000
ground_plane2.commit()
print(p)  # only 1 ground plane is taken per project, the second overwrites the previous one.

{
    "part_guid": "704b01a4-d8c0-4648-ba58-474c90b7f9fd",
    "sources": [
        {
            "name": "Environment.1",
            "metadata": {
                "UniqueId": "b5e637ba-3a8c-4733-b752-49a5b25349ee"
            },
            "source_guid": "0b41a621-3956-4c88-8751-f27282453d59",
            "display_name": "",
            "description": "",
            "source": {
                "name": "Environment.1",
                "ambient": {
                    "environment_map": {
                        "luminance": 1000.0,
                        "image_uri": "/app/assets/stars.exr",
                        "user_defined_rgb_space": {
                            "pre_defined_white_point": {
                                "white_point_type": "D50"
                            },
                            "red_spectrum_guid": "de4b4474-140e-4cb7-8cfd-ffae033e6199",
                            "green_spectrum_guid": "3e59d0c9-0f78-4a94-9e9f-f63c1fd9d763",
                   

{
    "part_guid": "704b01a4-d8c0-4648-ba58-474c90b7f9fd",
    "sources": [
        {
            "name": "Environment.1",
            "metadata": {
                "UniqueId": "b5e637ba-3a8c-4733-b752-49a5b25349ee"
            },
            "source_guid": "0b41a621-3956-4c88-8751-f27282453d59",
            "display_name": "",
            "description": "",
            "source": {
                "name": "Environment.1",
                "ambient": {
                    "environment_map": {
                        "luminance": 1000.0,
                        "image_uri": "/app/assets/stars.exr",
                        "user_defined_rgb_space": {
                            "pre_defined_white_point": {
                                "white_point_type": "D50"
                            },
                            "red_spectrum_guid": "de4b4474-140e-4cb7-8cfd-ffae033e6199",
                            "green_spectrum_guid": "3e59d0c9-0f78-4a94-9e9f-f63c1fd9d763",
                   

In [36]:
ground_plane.delete()
source6.delete()

### Display Source

In [37]:
d_src = p.create_source(name="Display.1", feature_type=SourceDisplay)
assert isinstance(d_src, SourceDisplay)
d_src.x_start = -6
d_src.x_end = 6
d_src.y_start = -5
d_src.y_end = 5
d_src.axis_system = [0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 1]
d_src.image_file_uri = assets_data_path / "test_display_source.1.speos" / "pyspeos.png"
d_src.commit()
print(d_src)

{
    "name": "Display.1",
    "metadata": {
        "UniqueId": "c706e463-5550-4490-aed6-56bee9363b32"
    },
    "source_guid": "6f91ee38-15d5-4bf3-bcbf-fd30cf78a4df",
    "display_name": "",
    "description": "",
    "source": {
        "name": "Display.1",
        "display": {
            "image_file_uri": "/app/assets/test_display_source.1.speos/pyspeos.png",
            "source_dimensions": {
                "x_start": -6.0,
                "x_end": 6.0,
                "y_start": -5.0,
                "y_end": 5.0
            },
            "luminous_flux": 50.0,
            "intensity_guid": "c5c251fb-754f-4525-a578-19db855028f2",
            "pre_defined_color_space": {
                "color_space_type": "sRGB"
            },
            "intensity": {
                "name": "Display.1.Intensity",
                "cos": {
                    "N": 1.0,
                    "total_angle": 180.0
                },
                "description": "",
                "metadata": {

In [38]:
d_src.delete()

### Thermic source

A thermic source emits radiation based on thermal properties of surfaces.
Two exitance modes are available: **emissive faces** (uniform temperature per face set)
and **temperature field** (temperature distribution read from a file).

#### Emissive faces mode

Create a thermic source and assign geometry faces with a uniform temperature.

In [39]:
source_thermic = p.create_source(name="Thermic.1", feature_type=SourceThermic)
print(source_thermic)  # local: default values before commit

local: {
    "name": "Thermic.1",
    "display_name": "",
    "description": "",
    "metadata": {},
    "source_guid": "",
    "source": {
        "name": "Thermic.1",
        "thermic": {
            "emissives_faces": {
                "temperature": 2000.0
            },
            "intensity_guid": ""
        },
        "description": "",
        "metadata": {}
    }
}


In [40]:
# Reuse geometry created earlier in the example (TheBodyD/TheFaceF)
thermic_face = p.find(name="TheBodyD/TheFaceF", feature_type=Face)[0]
thermic_face_op = p.create_optical_property(name="ThermicFaceOP.1")
thermic_face_op.set_volume_none()
thermic_face_op.set_surface_mirror().reflectance = 0
thermic_face_op.geometries = [thermic_face]
thermic_face_op.commit()
source_thermic.set_emissive_faces().geometries = [(thermic_face, False)]
source_thermic.set_emissive_faces().temperature = 3000  # temperature in K
source_thermic.commit()
print(source_thermic)

{
    "name": "Thermic.1",
    "metadata": {
        "UniqueId": "f94dc87a-5774-446e-ac13-1c8ecce2553e"
    },
    "source_guid": "d9383b0a-c885-4a68-81c9-a6da9e539dd5",
    "display_name": "",
    "description": "",
    "source": {
        "name": "Thermic.1",
        "thermic": {
            "emissives_faces": {
                "temperature": 3000.0
            },
            "intensity_guid": "006d8759-111a-4780-9512-0b07cba05c5b",
            "intensity": {
                "name": "Thermic.1.Intensity",
                "cos": {
                    "N": 1.0,
                    "total_angle": 180.0
                },
                "description": "",
                "metadata": {}
            }
        },
        "description": "",
        "metadata": {}
    }
}


In [41]:
# Increase temperature and recommit
source_thermic.set_emissive_faces().temperature = 5000
source_thermic.commit()
print(source_thermic)

{
    "name": "Thermic.1",
    "metadata": {
        "UniqueId": "f94dc87a-5774-446e-ac13-1c8ecce2553e"
    },
    "source_guid": "d9383b0a-c885-4a68-81c9-a6da9e539dd5",
    "display_name": "",
    "description": "",
    "source": {
        "name": "Thermic.1",
        "thermic": {
            "emissives_faces": {
                "temperature": 5000.0
            },
            "intensity_guid": "006d8759-111a-4780-9512-0b07cba05c5b",
            "intensity": {
                "name": "Thermic.1.Intensity",
                "cos": {
                    "N": 1.0,
                    "total_angle": 180.0
                },
                "description": "",
                "metadata": {}
            }
        },
        "description": "",
        "metadata": {}
    }
}


#### Temperature field mode

Switch the same source to temperature field mode. The temperature distribution
is read from an `.OPTTemperatureField` file. An axis plane positions the field
in the scene, and a surface optical property (SOP) controls reflectance.

In [42]:
temperature_field_file = assets_data_path / "TemperatureField_Tank.OPTTemperatureField"
source_thermic.set_temperature_field().temperature_field_uri = temperature_field_file
source_thermic.set_temperature_field().axis_plane = [0, 0, 0, 1, 0, 0, 0, 1, 0]
print(source_thermic.set_temperature_field().axis_plane)  # default axis plane

[0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 1.0, 0.0]


In [43]:
# Adjust SOP mirror reflectance for the temperature field surface
source_thermic.set_temperature_field().sop.set_surface_mirror().reflectance = 10
source_thermic.commit()
print(source_thermic)

{
    "name": "Thermic.1",
    "metadata": {
        "UniqueId": "f94dc87a-5774-446e-ac13-1c8ecce2553e"
    },
    "source_guid": "d9383b0a-c885-4a68-81c9-a6da9e539dd5",
    "display_name": "",
    "description": "",
    "source": {
        "name": "Thermic.1",
        "thermic": {
            "temperature_field": {
                "temperature_field_uri": "/app/assets/TemperatureField_Tank.OPTTemperatureField",
                "sop_guid": "1414ecbe-be59-4813-bcf7-38532b4c80bc",
                "sop": {
                    "name": "Thermic.1.SOP",
                    "mirror": {
                        "reflectance": 10.0
                    },
                    "description": "",
                    "metadata": {}
                },
                "axis_plane": [
                    0.0,
                    0.0,
                    0.0,
                    1.0,
                    0.0,
                    0.0,
                    0.0,
                    1.0,
                    0.

In [44]:
# Reset to server state and verify temperature field uri is preserved
source_thermic.reset()
print(source_thermic.set_temperature_field().temperature_field_uri)

/app/assets/TemperatureField_Tank.OPTTemperatureField


In [45]:
source_thermic.delete()

### Clean up

When creating sources, this creates some intermediate objects (spectrums, intensity templates).

Deleting a source does not delete in cascade those objects
because they could be used by some other entities from core layer.

Then at the end of the example, we just clean all databases

In [46]:
for item in speos.client.intensity_templates().list() + speos.client.spectrums().list():
    item.delete()

speos.close()

True